In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

d:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Step 1: Load and embed the document
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 30)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name  = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs  = {"k":4, "lambda_mult":0.7})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10250.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model("llama-3.1-8b-instant", model_provider="groq")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000216191DF770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021637BA72F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
## Step 3: Query Decomposition
decomposition_prompt  = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question:"{question}"
Sub-questions:""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [5]:
query  = "How does LangchChain use memory and agents compared to CrewAI?"
decomposition_question =  decomposition_chain.invoke({"question" : query})

In [6]:
print(decomposition_question)

Here are 4 smaller sub-questions that can be used to break down the complex question for better document retrieval:

1. "What is the architecture of LangChain's memory component?"
2. "How does LangChain's agent system differ from CrewAI's agent system?"
3. "What is the primary use of agents in LangChain compared to CrewAI?"
4. "How does LangChain's memory management compare to CrewAI's memory utilization?"

These sub-questions can help to retrieve relevant documents that provide information on the specific components of LangChain and CrewAI, allowing for a more in-depth comparison between the two systems.


In [7]:
## Step 4: QA Chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context: {context}

Question: {input}
""")


qa_chain = create_stuff_documents_chain(llm = llm, prompt = qa_prompt)

In [ ]:
## Step 5L Full RAG Pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text  = decomposition_chain.invoke({"question" : user_query})
    sub_questions = [q.strip("-1234567890. ") for q in sub_qs_text.split("\n") if q.strip()]

    results  =[]
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Question: {subq}\nAnswer: {result}")

    return "\n\n".join(results)